# AI Animation Studio — OverSimplified Style
Generates professional 2D animation from text prompts using Stable Diffusion + Ken Burns.
Runtime: ~3-5 min on T4 GPU. Free 30h/week on Kaggle.

In [ ]:
!pip install -q diffusers transformers accelerate safetensors edge-tts pillow
import torch, os, json, subprocess
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# ── CONFIG ──
TOPIC = 'gladiator'  # Change this for different videos!
FPS = 30
VOICE = 'en-US-ChristopherNeural'

SCENES = [
    {'scene':1, 'narration':'Why it sucks to be a gladiator.',
     'prompt':'flat vector illustration, ancient roman colosseum wide establishing shot, stone arena sand floor tiered seating crowd emperor box red curtains golden sunlight simple clean cartoon muted earth tones thick black outlines overhead angle digital art no text',
     'camera':'push-in', 'duration':8},
    {'scene':2, 'narration':'The colosseum. Thousands cheer as you enter the sand.',
     'prompt':'flat vector illustration, lone gladiator entering roman arena from dark gate, back view silhouette, massive colosseum walls towering, sand floor dramatic lighting, crowd silhouettes stands, muted earth tones thick black outlines simple cartoon dramatic cinematic digital art no text',
     'camera':'pan-left', 'duration':10},
    {'scene':3, 'narration':'Fighting to the death for the crowd entertainment.',
     'prompt':'flat vector illustration, two gladiators facing each other in roman arena, one with sword shield one with trident net, sand floor dramatic low angle, crowd cheering background, muted earth tones red accents thick black outlines simple cartoon dynamic composition digital art no text',
     'camera':'dolly-in', 'duration':10},
    {'scene':4, 'narration':'This is you. A slave with a sword. Fighting for your life.',
     'prompt':'flat vector illustration, close portrait roman gladiator, worn leather helmet red plume, determined expression simple dot eyes, thick black outlines muted earth tones, dark dramatic background simple cartoon emotional portrait digital art no text',
     'camera':'push-in-slow', 'duration':8},
]
STYLE = 'over simplified style, 2D flat vector art, thick uniform black outlines, solid color fills, simple character design, dot eyes, educational youtube animation, '
print(f'{len(SCENES)} scenes, ~{sum(s["duration"] for s in SCENES)}s total')

In [ ]:
# ── GENERATE ILLUSTRATIONS ──
from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler

pipe = StableDiffusionXLPipeline.from_pretrained(
    'stabilityai/stable-diffusion-xl-base-1.0',
    torch_dtype=torch.float16, variant='fp16', use_safetensors=True
)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe.to('cuda')
print('SDXL loaded on GPU')

os.makedirs('/kaggle/working/frames', exist_ok=True)
for s in SCENES:
    print(f'Generating scene {s["scene"]}...')
    img = pipe(
        prompt=STYLE + s['prompt'],
        negative_prompt='text, letters, words, watermark, blurry, realistic, photograph, 3d',
        width=1280, height=720, num_inference_steps=25,
        guidance_scale=7.5, generator=torch.Generator('cuda').manual_seed(42+s['scene'])
    ).images[0]
    img.save(f'/kaggle/working/frames/scene_{s["scene"]:02d}.png')
    print(f'  Saved scene {s["scene"]}')
del pipe; torch.cuda.empty_cache()

In [ ]:
# ── KEN BURNS + VOICEOVER + ASSEMBLY ──
os.makedirs('/kaggle/working/output', exist_ok=True)
os.makedirs('/kaggle/working/audio', exist_ok=True)

for s in SCENES:
    i = s['scene']
    print(f'Processing scene {i}: {s["camera"]} ({s["duration"]}s)')
    # Ken Burns
    scales = {'push-in':(1.0,1.25), 'push-in-slow':(1.0,1.15), 'pan-left':(1.15,1.15), 'dolly-in':(1.0,1.35)}
    sc = scales.get(s['camera'], (1.0,1.2))
    n = s['duration']*FPS
    subprocess.run(['ffmpeg','-y','-loop','1','-i',f'/kaggle/working/frames/scene_{i:02d}.png',
        '-vf',f'scale=5120:-1,zoompan=z={sc[0]}+(({sc[1]}-{sc[0]})/max(1,on))*on:d={n}:s=1280x720:fps={FPS}',
        '-frames:v',str(n), f'/kaggle/working/seg{i}.mp4'],
        capture_output=True)
    # Voiceover
    subprocess.run(['edge-tts','--voice',VOICE,'--text',s['narration'],
        '--write-media',f'/kaggle/working/audio/raw_{i:02d}.mp3'],
        capture_output=True)
    print(f'  Done scene {i}')

# Concat videos
with open('/kaggle/working/vlist.txt','w') as f:
    for s in SCENES:
        f.write(f"file 'seg{s['scene']}.mp4'\n")
subprocess.run(['ffmpeg','-y','-f','concat','-safe','0','-i','/kaggle/working/vlist.txt',
    '-c','copy','/kaggle/working/full_video.mp4'], capture_output=True)

# Concat audio
with open('/kaggle/working/alist.txt','w') as f:
    for s in SCENES:
        f.write(f"file 'audio/raw_{s['scene']:02d}.mp3'\n")
subprocess.run(['ffmpeg','-y','-f','concat','-safe','0','-i','/kaggle/working/alist.txt',
    '-c:a','aac','-b:a','192k','/kaggle/working/audio.mp4'], capture_output=True)

# Final merge
subprocess.run(['ffmpeg','-y','-i','/kaggle/working/full_video.mp4',
    '-i','/kaggle/working/audio.mp4','-c:v','copy','-c:a','aac','-shortest',
    '/kaggle/working/final.mp4'], capture_output=True)

size = os.path.getsize('/kaggle/working/final.mp4') if os.path.exists('/kaggle/working/final.mp4') else 0
print(f'\n✅ DONE: /kaggle/working/final.mp4 ({size//1024} KB)')

In [ ]:
# Download the final video
from IPython.display import FileLink
if os.path.exists('/kaggle/working/final.mp4'):
    print('Download: final.mp4')
    display(FileLink('/kaggle/working/final.mp4'))
else:
    print('No video found — check errors above')